# 17 Experiment 1 — Tokenization Analysis


Thống kê subword trên ít nhất 20 câu trong test set CoNLL-2003.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLP/lab09'
import os, json, random, numpy as np
os.makedirs(DRIVE_DIR, exist_ok=True)
SEED = 42
random.seed(SEED); np.random.seed(SEED)
!pip -q install datasets==3.6.0 transformers pandas
from datasets import load_dataset
from transformers import AutoTokenizer
import pandas as pd
ds=load_dataset('eriktks/conll2003', trust_remote_code=True)
label_names=ds['test'].features['ner_tags'].feature.names
tokenizer=AutoTokenizer.from_pretrained('bert-base-cased',use_fast=True)


In [ ]:
rows=[]
for sid,row in enumerate(ds['test'].select(range(20))):
    enc=tokenizer(row['tokens'],is_split_into_words=True,truncation=True,max_length=256)
    counts={i:0 for i in range(len(row['tokens']))}
    for wid in enc.word_ids():
        if wid is not None: counts[wid]+=1
    entities=set(i for i,t in enumerate(row['ner_tags']) if t!=0)
    for i,w in enumerate(row['tokens']): rows.append({'sentence_id':sid,'word':w,'word_count':len(row['tokens']),'subword_count':counts[i],'split':counts[i]>1,'entity_token':i in entities})
df=pd.DataFrame(rows)
print('Sentences:',df.sentence_id.nunique())
print(df.groupby('sentence_id').agg(words=('word','size'),tokens=('subword_count','sum')).assign(subword_ratio=lambda x:x.tokens/x.words).head(20).to_string())
print('By entity token:')
print(df.groupby('entity_token').agg(words=('word','size'),subword_tokens=('subword_count','sum'),split_rate=('split','mean')).to_string())
print('By sentence length:')
df['sentence_length']=pd.cut(df.word_count,[0,10,20,1000],labels=['short','medium','long'])
print(df.groupby('sentence_length',observed=True).agg(words=('word','size'),split_rate=('split','mean')).to_string())
df.to_csv(f'{DRIVE_DIR}/tokenization_analysis.csv',index=False)
